##Potrzebne biblioteki

In [ ]:
# Import generatora danych i podziału train/test
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# Import klasyfikatora drzewa decyzyjnego i metryk
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Import SMOTE do oversamplingu
from imblearn.over_sampling import SMOTE

# Import bibliotek do wizualizacji macierzy pomyłek
import seaborn as sns
import matplotlib.pyplot as plt


##Generowanie danych

In [ ]:
# Generowanie sztucznych danych binarnych
X, y = make_classification(
    # 2000 próbek
    n_samples=2000,
    # 10 cech
    n_features=10,
    # 4 cechy informatywne czyli takie dane co przenosza info potrzebne do uczenia modelu i pomagaja mu poprawnie rozroznic, klasyfikowac, przewisywac dane
    n_informative=4,
    #cechy miarowe - nie wprowadzaja nowego info, sa kombinacja innych cech. nie pomagaja lepiej uczyc modelu tylko powielaja istniejaca informacje.
    n_redundant=0,
    weights=[0.95, 0.05],
    #  brak szumu etykiet
    flip_y=0.0,
    # 95% klasy 0 i 5% klasy 1
    random_state=42
)

# Podział na zbiór treningowy i testowy
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    stratify=y,
    random_state=42
)

print("Liczność klas w zbiorze testowym:")
print(confusion_matrix(y_test, y_test))
print("Udział klasy 1 w teście:", y_test.mean())


##Rysuje macierz pomyłek czyli confiusion matrix
y_true - etykiety prawdziwe
y_pred - etykiety przewidziane

In [ ]:
def plot_cm(y_true, y_pred, title):

    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()


##Model 1 : Zwykłe drzewo bez balansowania





In [ ]:

# max_depth=1 - bardzo proste drzewo (jeden podział), które w tej konfiguracji nauczy się zawsze przewidywać klasę większościową 0.

m1 = DecisionTreeClassifier(max_depth=1, random_state=42)
# ucze drzewo na danych treningowych
m1.fit(X_train, y_train)
# przewiduje etykiety na zbiorze testowym
y_pred1 = m1.predict(X_test)

print("MODEL 1: Zwykłe drzewo (bez balansowania)")
print("Macierz pomyłek:")
print(confusion_matrix(y_test, y_pred1))
print("\nRaport klasyfikacji:")
print(classification_report(y_test, y_pred1))
print("Accuracy:", accuracy_score(y_test, y_pred1))

plot_cm(y_test, y_pred1, "Model 1: Bez balansowania")


##Wyniki:


*   570 prawdziwe 0 przewidzianne jako klasa 0
*   30 jedynek przewidzianych jako 0
*   0 przykladow wykrytych jako klasa 1 - rzadka

##Wniosek:
Model nigdy nie przywiduje klasy 1 zawsze bedzie 0. Dla klasy 0 metryki sa dobre. Model jest bezuzyteczny nie wykrywa rzadnego przypadku klasy 1.




##Model 2 : class_weight='balanced'

In [ ]:

# Tutaj mówie modelowi:błędy na klasie rzadkiej czyli 1 są ważniejsze niż błędy na klasie większościowej 0.

m2 = DecisionTreeClassifier(
     # głębsze drzewo
    max_depth=3,
    class_weight="balanced",
    random_state=42
)

m2.fit(X_train, y_train)
y_pred2 = m2.predict(X_test)

print("MODEL 2: Drzewo z class_weight='balanced'")
print("Macierz pomyłek:")
print(confusion_matrix(y_test, y_pred2))
print("\nRaport klasyfikacji:")
print(classification_report(y_test, y_pred2))
print("Accuracy:", accuracy_score(y_test, y_pred2))

plot_cm(y_test, y_pred2, "Model 2: class_weight='balanced'")


##Wyniki:


*   378 prawdziwe 0 przewidzianne jako klasa 0 poprawne
*   192 zer przewidzianych jako 1 blednie false positive klasa 1
*   9 przykladow wykrytych jako klasa 0 bledne false negative
*   21 jedynek jako klasa 1 poprawne true positive

##Wniosek:
Model bierze ze bledy na klasa 1 sa wazniejsze niz klasa 0. Z 21 przypadkow na 30 - recall 70 dla klasy rzadkiej. Dokladnosc spada z 95 na 66.5 %. Class_weight daje modelowi to że przestaje ignorowac klas mniejszosciowa.




##Model 3 : SMOTE + DRZEWO

In [ ]:
# SMOTE tworzy syntetyczne przykłady klasy 1, tak aby zrównoważyć zbiór.

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("Rozmiar zbioru przed SMOTE:", X_train.shape, "liczność klasy 1:", sum(y_train))
print("Rozmiar zbioru po SMOTE:   ", X_train_sm.shape, "liczność klasy 1:", sum(y_train_sm))

# Teraz ucze zwykle drzewo na danych po SMOTE
m3 = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

m3.fit(X_train_sm, y_train_sm)
y_pred3 = m3.predict(X_test)

print("MODEL 3: SMOTE + drzewo")
print("Macierz pomyłek:")
print(confusion_matrix(y_test, y_pred3))
print("\nRaport klasyfikacji:")
print(classification_report(y_test, y_pred3))
print("Accuracy:", accuracy_score(y_test, y_pred3))

plot_cm(y_test, y_pred3, "Model 3: SMOTE + drzewo")


##Wyniki:


*   373 zer prawdziwe jako klasa 0
*   197 zer blednie jako klasa 1
*   8 jedynek bledne jako klasa 0
*   22 jedynek poprawne jako klasa 1

##Wniosek:
Przed SMOTE:
W zbiorze treningowym bylo 1400 probek, w tym 70 czyli 5 % klasy rzadkiej 1
Po SMOTE:
SMOTE dodal sztuczne probki klasy 1 tak by liczba probek klasy 1 zrownala sie z klasa 0. Trening jest prowadzony na zbalansowanym w ten sposob zbiorze.




##Wnioski i obserwacje ogólne dla wszystkich 3 modeli:


1.   MODEL 1:
- Osiagnał wysoka dokladnosc - 95% jednak tak naprawde nie wykryl ani 1 przypadku klasy rzadkiej. Wynika to z tego, że w niezbalansowanych zbiorach model przewiduje to co mu się jakby "opłaca" przewidywac czyli klase wiekszosciowa. Wysoka dokladnosc przy niezbalansowanych jest mylaca i nie oddaje jakosci modelu.
2.   MODEL 2:
- Zastosowanie wag klas wymusilo na modelu silniejsze traktowanie bledow na klasie rzadkiej. Dzieki temu model wykryl 21 przypadkow z 30 klasy 1 - recall : 70 %. Class_weight znaczaco poprawia zdolnosc wykrywania klasy mnijszosciowej, lecz zmniejsza dokladnosc i zwieksza liczbe pomylek dla klasy wiekszosciowej.
3.   MODEL 3:
- Zbalansowanie zbioru treningowego przez SMOTE pozwlilo osoiagnac najwyzszy recall oraz nieco wyzsza wartosc F1-score w porowanianiu z class_weight. Wzorsla liczba jednak blednych klasyfikacji klasy 0. SMOTE przybliza model do wykrywania klasy rzadkiej co poprawia recall ale obniza ogolna dokladnosc i stablicnosc predykcji modelu.

##Przy danych niezbalansowanych kluczowe nie jest accuracy (dokładnośc), lecz zdolność modelu do wykrywania klas rzadkich, a najskuteczniej robią to metody balansowania: class_weight i SMOTE.


